In [1]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

print("✅ Imports loaded")

✅ Imports loaded


## Configuration

In [7]:
# Configuration
DATA_ROOT = "../data/movielenz_data"
FOLDS = list(range(1, 11))  # 1-10
VALIDATION_RATIO = 0.2  # 20% validation, 80% train_inner
RANDOM_SEED = 42

print("Configuration:")
print(f"  Folds: {FOLDS}")
print(f"  Validation ratio: {VALIDATION_RATIO*100:.0f}%")
print(f"  Random seed: {RANDOM_SEED}")
print(f"  Data root: {DATA_ROOT}")

Configuration:
  Folds: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Validation ratio: 20%
  Random seed: 42
  Data root: ../data/movielenz_data


## Helper Functions

In [8]:
def load_matrix_csv(path: str) -> pd.DataFrame:
    """Load CSV as (items × users) rating matrix."""
    df = pd.read_csv(path, low_memory=False)
    
    # Check first column name
    first_col = df.columns[0]
    if str(first_col).lower() in ("item", "items", "item_id", "id", "index", "unnamed: 0"):
        df = df.set_index(first_col)
    
    # Clear names
    df.index.name = None
    df.columns.name = None
    
    # Convert to numeric
    df = df.apply(pd.to_numeric, errors="coerce")
    
    # Sort by integer index/columns
    df.index = df.index.astype(int)
    df.columns = df.columns.astype(int)
    df = df.sort_index().sort_index(axis=1)
    
    return df


def save_matrix_csv(df: pd.DataFrame, path: str):
    """Save matrix to CSV without index/column names."""
    df_save = df.copy()
    df_save.index.name = None
    df_save.columns.name = None
    df_save.to_csv(path, index=True)
    

def split_train_user_wise(train_matrix: pd.DataFrame, 
                          validation_ratio: float = 0.2,
                          random_seed: int = 42) -> tuple:
    """
    Split train matrix into train_inner and validation using user-wise random sampling.
    
    Args:
        train_matrix: (items × users) rating matrix
        validation_ratio: fraction of each user's ratings to holdout
        random_seed: random seed for reproducibility
    
    Returns:
        (train_inner, validation) both as DataFrames
    """
    np.random.seed(random_seed)
    
    train_inner = train_matrix.copy()
    validation = train_matrix.copy()
    
    # Initialize as all NaN
    train_inner[:] = np.nan
    validation[:] = np.nan
    
    total_ratings = 0
    val_ratings = 0
    
    # For each user (column)
    for user_idx in range(train_matrix.shape[1]):
        user_col = train_matrix.iloc[:, user_idx]
        user_id = train_matrix.columns[user_idx]
        observed_mask = ~user_col.isna()
        observed_items = user_col[observed_mask].index.tolist()
        
        n_observed = len(observed_items)
        if n_observed == 0:
            continue
        
        total_ratings += n_observed
        
        # Determine validation size (at least 1 if possible)
        n_val = max(1, int(np.round(n_observed * validation_ratio)))
        n_val = min(n_val, n_observed - 1)  # Keep at least 1 in train_inner
        
        # Random sample
        val_items = np.random.choice(observed_items, size=n_val, replace=False)
        train_items = [item for item in observed_items if item not in val_items]
        
        # Assign (use .loc for label-based indexing)
        for item in train_items:
            train_inner.loc[item, user_id] = train_matrix.loc[item, user_id]
        
        for item in val_items:
            validation.loc[item, user_id] = train_matrix.loc[item, user_id]
        
        val_ratings += len(val_items)
    
    print(f"  Total ratings: {total_ratings:,}")
    print(f"  Train_inner: {(~train_inner.isna()).sum().sum():,} ({(~train_inner.isna()).sum().sum()/total_ratings*100:.1f}%)")
    print(f"  Validation: {(~validation.isna()).sum().sum():,} ({(~validation.isna()).sum().sum()/total_ratings*100:.1f}%)")
    
    return train_inner, validation

print("✅ Helper functions defined")

✅ Helper functions defined


## Main Processing

In [9]:
print("="*80)
print("CREATING TRAIN/VALIDATION SPLITS")
print("="*80)

for fold_num in FOLDS:
    print(f"\n{'='*80}")
    print(f"FOLD {fold_num:02d}")
    print("="*80)
    
    fold_dir = os.path.join(DATA_ROOT, f"fold_{fold_num:02d}")
    
    # Load train matrix
    train_path = os.path.join(fold_dir, "train.csv")
    print(f"Loading: {train_path}")
    train_matrix = load_matrix_csv(train_path)
    print(f"Shape: {train_matrix.shape} (items × users)")
    
    # Split
    print(f"\nSplitting (validation={VALIDATION_RATIO*100:.0f}%, seed={RANDOM_SEED})...")
    train_inner, validation = split_train_user_wise(
        train_matrix,
        validation_ratio=VALIDATION_RATIO,
        random_seed=RANDOM_SEED
    )
    
    # Save
    train_inner_path = os.path.join(fold_dir, "train_inner.csv")
    validation_path = os.path.join(fold_dir, "validation.csv")
    
    print(f"\nSaving:")
    print(f"  {train_inner_path}")
    save_matrix_csv(train_inner, train_inner_path)
    print(f"  {validation_path}")
    save_matrix_csv(validation, validation_path)
    
    # Verify
    print(f"\nVerification:")
    train_vals = set()
    val_vals = set()
    
    for i in range(train_inner.shape[0]):
        for j in range(train_inner.shape[1]):
            if not pd.isna(train_inner.iloc[i, j]):
                train_vals.add((i, j, train_inner.iloc[i, j]))
            if not pd.isna(validation.iloc[i, j]):
                val_vals.add((i, j, validation.iloc[i, j]))
    
    overlap = train_vals & val_vals
    print(f"  Overlap count: {len(overlap)} (should be 0)")
    print(f"  Union count: {len(train_vals | val_vals)}")
    print(f"  Original count: {(~train_matrix.isna()).sum().sum()}")
    
    if len(overlap) > 0:
        print(f"  ⚠️ WARNING: Found {len(overlap)} overlapping ratings!")
    else:
        print(f"  ✅ No overlap - split is valid")
    
    print(f"✅ Fold {fold_num:02d} complete")

print(f"\n{'='*80}")
print("ALL SPLITS COMPLETE")
print("="*80)

CREATING TRAIN/VALIDATION SPLITS

FOLD 01
Loading: ../data/movielenz_data\fold_01\train.csv
Shape: (1682, 943) (items × users)

Splitting (validation=20%, seed=42)...
  Total ratings: 89,573
  Train_inner: 71,637 (80.0%)
  Validation: 17,936 (20.0%)

Saving:
  ../data/movielenz_data\fold_01\train_inner.csv
  ../data/movielenz_data\fold_01\validation.csv

Verification:
  Overlap count: 0 (should be 0)
  Union count: 89573
  Original count: 89573
  ✅ No overlap - split is valid
✅ Fold 01 complete

FOLD 02
Loading: ../data/movielenz_data\fold_02\train.csv
Shape: (1682, 943) (items × users)

Splitting (validation=20%, seed=42)...
  Total ratings: 89,680
  Train_inner: 71,721 (80.0%)
  Validation: 17,959 (20.0%)

Saving:
  ../data/movielenz_data\fold_02\train_inner.csv
  ../data/movielenz_data\fold_02\validation.csv

Verification:
  Overlap count: 0 (should be 0)
  Union count: 89680
  Original count: 89680
  ✅ No overlap - split is valid
✅ Fold 02 complete

FOLD 03
Loading: ../data/moviele

## Summary

In [10]:
print("Summary of created files:")
print("="*80)

for fold_num in FOLDS:
    fold_dir = os.path.join(DATA_ROOT, f"fold_{fold_num:02d}")
    train_inner_path = os.path.join(fold_dir, "train_inner.csv")
    validation_path = os.path.join(fold_dir, "validation.csv")
    
    if os.path.exists(train_inner_path) and os.path.exists(validation_path):
        train_inner = load_matrix_csv(train_inner_path)
        validation = load_matrix_csv(validation_path)
        
        print(f"Fold {fold_num:02d}:")
        print(f"  train_inner: {(~train_inner.isna()).sum().sum():,} ratings")
        print(f"  validation:  {(~validation.isna()).sum().sum():,} ratings")
    else:
        print(f"Fold {fold_num:02d}: ⚠️ Missing files")

print("="*80)

Summary of created files:
Fold 01:
  train_inner: 71,637 ratings
  validation:  17,936 ratings
Fold 02:
  train_inner: 71,721 ratings
  validation:  17,959 ratings
Fold 03:
  train_inner: 71,799 ratings
  validation:  17,974 ratings
Fold 04:
  train_inner: 71,894 ratings
  validation:  17,991 ratings
Fold 05:
  train_inner: 71,976 ratings
  validation:  17,998 ratings
Fold 06:
  train_inner: 72,041 ratings
  validation:  18,031 ratings
Fold 07:
  train_inner: 72,117 ratings
  validation:  18,049 ratings
Fold 08:
  train_inner: 72,189 ratings
  validation:  18,073 ratings
Fold 09:
  train_inner: 72,259 ratings
  validation:  18,089 ratings
Fold 10:
  train_inner: 72,319 ratings
  validation:  18,094 ratings


## Configuration

In [2]:
# Settings
FOLDS = list(range(1, 11))  # fold_01 to fold_10
VALIDATION_RATIO = 0.2  # 20% for validation
RANDOM_SEED = 42

# Paths
DATA_ROOT = "../data/movielenz_data"

print(f"Configuration:")
print(f"  Folds: {FOLDS}")
print(f"  Validation ratio: {VALIDATION_RATIO*100:.0f}%")
print(f"  Random seed: {RANDOM_SEED}")
print(f"  Data root: {DATA_ROOT}")

Configuration:
  Folds: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Validation ratio: 20%
  Random seed: 42
  Data root: ../data/movielenz_data


## Helper Functions

In [5]:
def load_matrix_csv(path: str) -> pd.DataFrame:
    """Load CSV as (items × users) rating matrix."""
    df = pd.read_csv(path, low_memory=False)
    
    # Check first column name
    first_col = df.columns[0]
    if str(first_col).lower() in ("item", "items", "item_id", "id", "index", "unnamed: 0"):
        df = df.set_index(first_col)
    
    # Clear names
    df.index.name = None
    df.columns.name = None
    
    # Convert to numeric
    df = df.apply(pd.to_numeric, errors="coerce")
    
    # Sort by integer index/columns
    df.index = df.index.astype(int)
    df.columns = df.columns.astype(int)
    df = df.sort_index().sort_index(axis=1)
    
    return df


def save_matrix_csv(df: pd.DataFrame, path: str):
    """Save matrix to CSV without index/column names."""
    df_save = df.copy()
    df_save.index.name = None
    df_save.columns.name = None
    df_save.to_csv(path, index=True)
    

def split_train_user_wise(train_matrix: pd.DataFrame, 
                          validation_ratio: float = 0.2,
                          random_seed: int = 42) -> tuple:
    """
    Split train matrix into train_inner and validation using user-wise random sampling.
    
    Args:
        train_matrix: (items × users) rating matrix
        validation_ratio: fraction of each user's ratings to holdout
        random_seed: random seed for reproducibility
    
    Returns:
        (train_inner, validation) both as DataFrames
    """
    np.random.seed(random_seed)
    
    train_inner = train_matrix.copy()
    validation = train_matrix.copy()
    
    # Initialize as all NaN
    train_inner[:] = np.nan
    validation[:] = np.nan
    
    total_ratings = 0
    val_ratings = 0
    
    # For each user (column)
    for user_idx in range(train_matrix.shape[1]):
        user_col = train_matrix.iloc[:, user_idx]
        observed_mask = ~user_col.isna()
        observed_items = user_col[observed_mask].index.tolist()
        
        n_observed = len(observed_items)
        if n_observed == 0:
            continue
        
        total_ratings += n_observed
        
        # Determine validation size (at least 1 if possible)
        n_val = max(1, int(np.round(n_observed * validation_ratio)))
        n_val = min(n_val, n_observed - 1)  # Keep at least 1 in train_inner
        
        # Random sample
        val_items = np.random.choice(observed_items, size=n_val, replace=False)
        train_items = [item for item in observed_items if item not in val_items]
        
        # Assign
        for item in train_items:
            train_inner.iloc[item, user_idx] = train_matrix.iloc[item, user_idx]
        
        for item in val_items:
            validation.iloc[item, user_idx] = train_matrix.iloc[item, user_idx]
        
        val_ratings += len(val_items)
    
    print(f"  Total ratings: {total_ratings:,}")
    print(f"  Train_inner: {(~train_inner.isna()).sum().sum():,} ({(~train_inner.isna()).sum().sum()/total_ratings*100:.1f}%)")
    print(f"  Validation: {(~validation.isna()).sum().sum():,} ({(~validation.isna()).sum().sum()/total_ratings*100:.1f}%)")
    
    return train_inner, validation

print("✅ Helper functions defined")

✅ Helper functions defined


## Process All Folds

In [6]:
print("="*80)
print("CREATING TRAIN/VALIDATION SPLITS")
print("="*80)

for fold_num in FOLDS:
    fold_id = f"fold_{fold_num:02d}"
    fold_dir = os.path.join(DATA_ROOT, fold_id)
    train_path = os.path.join(fold_dir, "train.csv")
    
    if not os.path.exists(train_path):
        print(f"\n⚠️  Skipping {fold_id}: train.csv not found")
        continue
    
    print(f"\n{'='*80}")
    print(f"FOLD {fold_num:02d}")
    print(f"{'='*80}")
    
    # Load train matrix
    print(f"Loading: {train_path}")
    train_matrix = load_matrix_csv(train_path)
    print(f"Shape: {train_matrix.shape} (items × users)")
    
    # Split
    print(f"\nSplitting (validation={VALIDATION_RATIO*100:.0f}%, seed={RANDOM_SEED})...")
    train_inner, validation = split_train_user_wise(
        train_matrix,
        validation_ratio=VALIDATION_RATIO,
        random_seed=RANDOM_SEED
    )
    
    # Save
    train_inner_path = os.path.join(fold_dir, "train_inner.csv")
    validation_path = os.path.join(fold_dir, "validation.csv")
    
    print(f"\nSaving...")
    save_matrix_csv(train_inner, train_inner_path)
    print(f"  ✓ {train_inner_path}")
    
    save_matrix_csv(validation, validation_path)
    print(f"  ✓ {validation_path}")
    
    # Verify
    print(f"\nVerification:")
    print(f"  Train_inner min/max: {train_inner.min().min():.1f} / {train_inner.max().max():.1f}")
    print(f"  Validation min/max: {validation.min().min():.1f} / {validation.max().max():.1f}")
    
    # Check no overlap
    overlap = (~train_inner.isna()) & (~validation.isna())
    if overlap.any().any():
        print(f"  ⚠️  WARNING: {overlap.sum().sum()} overlapping ratings!")
    else:
        print(f"  ✓ No overlap between train_inner and validation")

print(f"\n{'='*80}")
print("✅ ALL FOLDS PROCESSED")
print(f"{'='*80}")
print(f"\nNext steps:")
print(f"  1. Use train_inner.csv for KNN learning during α optimization")
print(f"  2. Use validation.csv for α evaluation/selection")
print(f"  3. Use full train.csv for final test evaluation")
print(f"  4. Use test.csv ONLY for final evaluation (never during α optimization)")

CREATING TRAIN/VALIDATION SPLITS

FOLD 01
Loading: ../data/movielenz_data\fold_01\train.csv
Shape: (1682, 943) (items × users)

Splitting (validation=20%, seed=42)...


IndexError: index 1682 is out of bounds for axis 0 with size 1682